# Contextual Alertness — Midterm Project
**FINM 36700 · Portfolio and Risk Management · Autumn 2026**

**Group:** HW Group B — Avi Rastogi, Lucas Albin Lazcano, Mengting Wang, Siena Verprauskus

**Use of AI:** Generative AI (Claude) was used for notebook scaffolding, code drafting and debugging, and review of interpretation in Sections _X–Y_. All results and judgments were checked by the group.

## Project Statement

At the December 31, 2024 close a fictional long-short equity fund, Contextual Alertness, holds six memory, optical-networking and data-center names against six application-software names: 400% gross, 0% net, equal weights within each sleeve. The book is a stylized stand-in inspired by a real fund's publicly reported positioning. It is not that fund's holdings. Over the next eighteen months the book rises many-fold, then loses most of that gain in four sessions, July 24 to 29, 2026.

You are the risk manager. Take the book as given. Measure its risk, locate where the risk sat, say what a one-day VaR could and could not have told you, and build the version of the same strategy you would have been willing to run.

## Context

| Long, +1/3 of NAV each | Theme | Short, -1/3 of NAV each | Theme |
|---|---|---|---|
| `SNDK` | memory and storage | `ADBE` | application software |
| `MU` | memory | `CRM` | application software |
| `VRT` | data-center power and cooling | `ORCL` | enterprise software |
| `LITE` | optical networking | `NOW` | enterprise software |
| `COHR` | optical networking | `WDAY` | application software |
| `BE` | data-center power | `MDB` | database software |

The long sleeve is +200% of NAV and the short sleeve -200%, so gross exposure is 400% and net exposure is zero. The `SNDK` column is one strategy slot: it carries `WDC` returns through February 21, 2025 and `SNDK` returns from February 24, 2025, when Sandisk began regular-way trading; the switch is already in the supplied series.

The fund runs a mandate. At every close the book is reset to the target weights, so gross stays at 4x and net at zero: the **4x book**. Dividing every weight by four gives the **1x book**, the same positions at 100% gross.

Everything through July 31, 2026 is yours to study. The holdout, August 3 to October 26, 2026, exists for one purpose: Section 6 has you declare criteria, build controls on the design window, then run them, unchanged, on the holdout. Do not tune anything to it. If you study it earlier, the test in 6.3 stops measuring anything.

## Data

| File | Sheets | Frequency | Sample |
|---|---|---|---|
| `data/project_situational_awareness_20261026.xlsx` | `descriptions`, `total return indexes`, `total returns`, `baseline weights`, `data source`, `checks` | daily | 2020-12-31 to 2026-10-26 |

The `total returns` sheet holds the twelve names, `SPY`, `QQQ`, `SMH`, the eleven sector ETFs (`XLK XLI XLF XLC XLRE XLE XLY XLB XLV XLU XLP`) and `SHV`, as gross-dividend total returns. `descriptions` gives each ticker's name, theme and side; `baseline weights` gives the twelve target weights.

### Notes

- Returns are daily, close to close, simple total returns. A weight set at the close of $t$ earns the return of $t+1$: $r^{\text{book}}_{t+1} = \boldsymbol{w}_t' \boldsymbol{r}_{t+1}$. Annualize with 252 trading days.
- The daily reset is a strategy-index convention: resetting to the target weights at every close makes 4x the book's exposure on every day, so Sections 3 to 5 measure the risk of a fixed-leverage book. Section 2 shows what the convention assumes away.
- Cash is `SHV`. Names and factors enter every regression in excess of `SHV`. The book's weights sum to zero, so its return $\boldsymbol{w}'\boldsymbol{r}$ is already an excess return and its Sharpe ratio uses it directly; the fund's NAV earns `SHV` on top of it.
- VaR and CVaR are positive loss magnitudes, one-day at the 5% level unless a question says otherwise. A forecast for day $t$ uses data through $t-1$. A breach is a day whose realized loss exceeds the VaR forecast made for it. EWMA uses $\lambda = 0.94$ and a zero mean; you choose how to initialize $\sigma^2$ and state it.
- The **design window** ends July 31, 2026. The **report window** is January 2, 2025 to July 31, 2026. The **holdout** runs from August 3, 2026 to October 26, 2026, the end of the file. Use the design window for all estimation and design. Open the holdout once, in 6.3, to evaluate the controls and criteria already specified. The final recommendation in 7.4 may use that verdict; do not re-tune the controls on the holdout.
- The **four-day loss** is the compounded loss of the daily-reset 4x book over July 24 to 29, 2026, $1 - \prod_{t=1}^{4}(1 + r_t)$, with $r_t$ the book's daily return.
- Ignore financing, stock-borrow fees, transaction costs, taxes and market impact.

## Setup

In [29]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

DATA = Path('../data') / 'project_situational_awareness_20261026.xlsx'

weights     = pd.read_excel(DATA, sheet_name='baseline weights').set_index('ticker')['equal-sleeve weight']
descriptions = pd.read_excel(DATA, sheet_name='descriptions').set_index('ticker')
returns_all = pd.read_excel(DATA, sheet_name='total returns', index_col='Date', parse_dates=True)

DESIGN_END = pd.Timestamp('2026-07-31')
returns = returns_all.loc[:DESIGN_END]                              #most questions use returns up to 31st of july, except 6.1

# Windows
FULL     = slice('2021-01-04', '2026-06-30')
RECENT   = slice('2025-01-02', '2026-06-30')
REPORT   = slice('2025-01-02', '2026-07-31')
REVERSAL = slice('2026-07-24', '2026-07-29')

# Tickers
tickers       = weights.index.tolist()
long_tickers  = weights[weights > 0].index.tolist()
short_tickers = weights[weights < 0].index.tolist()

# Books (daily reset): already excess returns since weights sum to zero
book_4x = returns[tickers] @ weights
book_1x = book_4x / 4
rf = returns['SHV']

In [30]:
descriptions

,name,theme,side
ticker,,,
SNDK,Sandisk,memory and storage,long
MU,Micron Technology,memory,long
VRT,Vertiv,data-center power and cooling,long
LITE,Lumentum,optical networking,long
COHR,Coherent,optical networking,long
BE,Bloom Energy,data-center power,long
ADBE,Adobe,application software,short
CRM,Salesforce,application software,short
ORCL,Oracle,enterprise software,short


In [31]:
returns.describe().T

,count,mean,std,min,25%,50%,75%,max
SNDK,1400.0,0.003257,0.040940,-0.213016,-0.016529,0.000748,0.018967,0.275649
MU,1400.0,0.002326,0.034687,-0.161788,-0.016089,0.001007,0.019716,0.192916
VRT,1400.0,0.002582,0.038244,-0.367399,-0.016441,0.002631,0.022803,0.292498
LITE,1400.0,0.002170,0.038032,-0.224569,-0.015780,0.001249,0.019684,0.235666
COHR,1400.0,0.001657,0.039108,-0.298660,-0.017891,0.001166,0.022179,0.228531
BE,1400.0,0.002877,0.055240,-0.247903,-0.029408,0.000000,0.031284,0.591867
ADBE,1400.0,-0.000228,0.022894,-0.167932,-0.010812,0.000164,0.012015,0.145115
CRM,1400.0,0.000145,0.023107,-0.197371,-0.011429,0.000264,0.012010,0.114969
ORCL,1400.0,0.000887,0.026163,-0.137908,-0.010157,0.000868,0.011370,0.359488
NOW,1400.0,0.000388,0.027564,-0.177452,-0.013072,0.001536,0.014373,0.154854


In [32]:
weights

ticker
SNDK    0.333333
MU      0.333333
VRT     0.333333
LITE    0.333333
COHR    0.333333
BE      0.333333
ADBE   -0.333333
CRM    -0.333333
ORCL   -0.333333
NOW    -0.333333
WDAY   -0.333333
MDB    -0.333333
Name: equal-sleeve weight, dtype: float64

---
# 1. The Book as Given

## 1.1

> Verify the long, short, gross and net exposures as multiples of NAV and as dollars on $100 million of NAV. Then, for the 4x book and the 1x book, report over 2021-01-04 to 2026-06-30 and over 2025-01-02 to 2026-06-30:
> 
> - annualized mean and volatility, Sharpe ratio, skewness, excess kurtosis
> - worst day, empirical 5% VaR and CVaR
> - maximum drawdown with its peak and trough dates, and cumulative return
> 
> Group the statistics by how they respond to leverage. Explain the scaling and compounding within each group.

In [33]:
longs = weights[long_tickers]
shorts = weights[short_tickers]

long_exposure = longs.sum()
short_exposure = shorts.sum()

gross_exposure = long_exposure + abs(short_exposure)
net_exposure = long_exposure + short_exposure

NAV = 100

exposures = pd.DataFrame({
    'x NAV': [long_exposure, short_exposure, gross_exposure, net_exposure]
}, index=['Long', 'Short', 'Gross', 'Net'])

exposures['$M on $100M'] = exposures['x NAV'] * NAV
exposures.style.format({'x NAV': '{:.1f}', '$M on $100M': '{:,.0f}'})

,x NAV,$M on $100M
Long,2.0,200
Short,-2.0,-200
Gross,4.0,400
Net,0.0,0


_Answer:_ Exposures check out: long +2.0x ($200M), short −2.0x (−$200M), gross 4.0x ($400M), net 0.0x ($0) on $100M of NAV.

In [34]:
def path_stats(r):
    wealth = (1 + r).cumprod()
    dd = wealth / wealth.cummax() - 1
    trough = dd.idxmin()
    peak = wealth.loc[:trough].idxmax()

    return dd.min(), peak.date(), trough.date(), wealth.iloc[-1] - 1


def stats(r):
    ann_mean = r.mean() * 252
    ann_vol = r.std() * np.sqrt(252)
    sharpe = ann_mean / ann_vol
    skewness = r.skew()
    excess_kurt = r.kurt()
    
    worst_day = min(r)
    emp_VaR = -r.quantile(0.05)
    emp_CVaR = -r[r <= -emp_VaR].mean()
    
    max_drawdown, peak_date, trough_date, cum_return = path_stats(r)

    return pd.Series({
        'Ann. mean': ann_mean,
        'Ann. vol': ann_vol,
        'Sharpe': sharpe,
        'Skewness': skewness,
        'Excess Kurtosis': excess_kurt,
        "Worst Day": worst_day,
        "VaR": emp_VaR,
        "CVaR": emp_CVaR,
        "Max Drawdown": max_drawdown,
        "Peak": peak_date,
        "Trough": trough_date,
        "Cumulative Return": cum_return
    })

summary_stats = pd.DataFrame({"4x, 2021–2026": stats(book_4x[FULL]),
                            "4x, 2025–2026": stats(book_4x[RECENT]),
                            "1x, 2021–2026": stats(book_1x[FULL]),
                            "1x, 2025–2026": stats(book_1x[RECENT])})

pct_stats = ['Ann. mean', 'Ann. vol', 'Worst Day', 'VaR', 'CVaR', 'Max Drawdown', 'Cumulative Return']
ratio_stats = ['Sharpe', 'Skewness', 'Excess Kurtosis']

summary_stats.style.format('{:.2%}', subset=(pct_stats, summary_stats.columns)) \
                   .format('{:.2f}', subset=(ratio_stats, summary_stats.columns))

,"4x, 2021–2026","4x, 2025–2026","1x, 2021–2026","1x, 2025–2026"
Ann. mean,124.72%,424.81%,31.18%,106.20%
Ann. vol,89.87%,136.64%,22.47%,34.16%
Sharpe,1.39,3.11,1.39,3.11
Skewness,0.07,-0.34,0.07,-0.34
Excess Kurtosis,4.64,1.80,4.64,1.80
Worst Day,-35.96%,-35.96%,-8.99%,-8.99%
VaR,7.32%,13.63%,1.83%,3.41%
CVaR,12.35%,19.75%,3.09%,4.94%
Max Drawdown,-79.38%,-48.44%,-26.26%,-13.57%
Peak,2021-01-19,2026-05-13,2021-01-19,2026-05-13


In [35]:
num = summary_stats.drop(['Peak', 'Trough']).astype(float)
ratio = num['4x, 2021–2026'] / num['1x, 2021–2026']

print('Scales x4: ', list(ratio[np.isclose(ratio, 4)].index))
print('Invariant: ', list(ratio[np.isclose(ratio, 1)].index))
print('Non-linear:', list(ratio[~np.isclose(ratio, 4) & ~np.isclose(ratio, 1)].index))

Scales x4:  ['Ann. mean', 'Ann. vol', 'Worst Day', 'VaR', 'CVaR']
Invariant:  ['Sharpe', 'Skewness', 'Excess Kurtosis']
Non-linear: ['Max Drawdown', 'Cumulative Return']


**Answer 1.1.**

The descriptive statisticss respond to leverage in three different ways: they either scale by 4x, are invariant, or respond non-linearly. 

Annual mean, annual volatility, worst day, and empirical VaR and CVaR all scale linearly with leverage. This is expected, as they all depend on the size of each day's return, and all daily returns are scaled by 4x with leverage.

Sharpe, skewness and excess kurtosis remain equal across leverage levels. Sharpe remains the same because leverage amplifies returns but equally amplifies volatility. Skewness and excess kurtosis are characteristics of the daily return distribution which are unaffected by scaling, since they are standardized moments.

Finally, and most interestingly, maximum drawdown and cumulative returns are non-linear in leverage. Maximum drawdown goes from −26.3% to −79.4% (ratio 3.02) over 2021–2026. The reason it can't scale linearly because it is bounded by −100%, as the NAV cannot fall below 0, and because it is measured on the compounded path. Cumulative returns do not scale linearly because they are path-dependent and compound. Over 2021–2026 the 4x book returned 9,805% vs. 379% for the 1x book (ratio 25.9), and over 2025–2026 12,690% vs. 340% (ratio 37.3). 

## 1.2

> Report the compounded calendar-year return of the 4x book for 2021 through 2026 (to July 31), with the contribution of each sleeve, defined as the sum of the sleeve's daily weighted returns. The sleeve contributions add to the sum of the book's daily returns rather than to its compounded year; report both. In which years did the short sleeve add to the return, and in which did it subtract?

In [36]:
def sleeve_returns_yearly(r):
    book  = r[tickers] @ weights
    long  = r[long_tickers] @ weights[long_tickers]
    short = r[short_tickers] @ weights[short_tickers]

    return pd.Series({
        "Compounded book return": (1 + book).prod() - 1,
        "Sum of daily book returns": book.sum(),
        "Long sleeve contribution": long.sum(),
        "Short sleeve contribution": short.sum(),
    })

decomposed_returns = pd.DataFrame({"2021": sleeve_returns_yearly(returns.loc['2021']),
"2022": sleeve_returns_yearly(returns.loc['2022']),
"2023": sleeve_returns_yearly(returns.loc['2023']),
"2024": sleeve_returns_yearly(returns.loc['2024']),
"2025": sleeve_returns_yearly(returns.loc['2025']),
"2026": sleeve_returns_yearly(returns.loc['2026'])})

decomposed_returns.T.style.format('{:.1%}')

,Compounded book return,Sum of daily book returns,Long sleeve contribution,Short sleeve contribution
2021,-28.9%,-19.4%,36.6%,-55.9%
2022,-20.7%,-3.8%,-79.1%,75.3%
2023,-34.8%,-25.1%,100.5%,-125.6%
2024,110.6%,101.5%,123.5%,-22.0%
2025,795.5%,274.6%,288.8%,-14.2%
2026,325.8%,267.5%,223.7%,43.9%


**Answer 1.2.**

The short sleeve was negative in 2021, 2023, 2024, and 2025, subtracting from total returns. It contributed to negative returns in 2021 and 2023, with a significant rally in 2023, as its losses exceeded the long sleeve's gains. The only years in which the short sleeve added to returns were 2022, where it contributed a sum of daily returns of 75%, and 2026 (+44%). 2022 is the only year it actually hedged, as in 2026 both sleeves gained.

The long sleeve returns were positive for all years except 2022, where it underperformed the short sleeve and thus the sum of daily book returns was negative. In all other years, though, the long sleeve was tremendously profitable. The continuous rallies from 2024 through 2026, with sums of daily returns of 124% to 289%, drove the sum of daily book returns as the short sleeve losses were relatively smaller (and in 2026 the short sleeve added as well).

Comparing the compounded book return with the sum of daily book returns we see a significant difference between 2025 and 2026. Noting that 2026 only contains 7 months of returns, we see that the sum of daily book returns in both time periods is almost equal (274.6% in 2025 vs 267.5% in the first 7 months of 2026), whereas the compounded book return in 2025 is almost 2.5 times as large as that of 2026 (795.5% vs 325.8%). This reveals the fact that the path of returns matters: indeed, the July loss was such that it wiped out about 62% of the year to date gains (the book was up 863% through July 23 and only 326% by July 31), which is something the sum of daily returns does not show.

In [37]:
jul_23 = (1 + book_4x.loc['2026-01-01':'2026-07-23']).prod() - 1 
jul_31 = (1 + book_4x.loc['2026']).prod() - 1 

gain_erased = (jul_23 - jul_31) / jul_23

print("Jul 23: ", round(jul_23,2))
print("Jul 31: ", round(jul_31,2))

print("Gain erased:", round(gain_erased,2))

Jul 23:  8.63
Jul 31:  3.26
Gain erased: 0.62


## 1.3

> Over both windows in 1.1, report
> 
> - the average pairwise correlation among the long names, among the short names, and between a long and a short name
> - the correlation between the long basket's return and the short basket's return
> 
> What, if anything, does the short sleeve hedge?

In [38]:
def avg_corr(df):
    c = df.corr().values
    return c[np.triu_indices_from(c, k=1)].mean()

def corr_stats(r):
    c = r[tickers].corr()
    long_basket  = r[long_tickers].mean(axis=1)
    short_basket = r[short_tickers].mean(axis=1)
    return pd.Series({
        'Avg corr: long-long':   avg_corr(r[long_tickers]),
        'Avg corr: short-short': avg_corr(r[short_tickers]),
        'Avg corr: long-short':  c.loc[long_tickers, short_tickers].values.mean(),
        'Corr: long basket vs short basket': long_basket.corr(short_basket),
    })

corr_table = pd.DataFrame({'2021–2026': corr_stats(returns.loc[FULL]),
                           '2025–2026': corr_stats(returns.loc[RECENT])})
corr_table.style.format('{:.2f}')

,2021–2026,2025–2026
Avg corr: long-long,0.48,0.55
Avg corr: short-short,0.50,0.50
Avg corr: long-short,0.24,0.11
Corr: long basket vs short basket,0.43,0.23


**Answer 1.3.**

The average correlations above show that within the sleeves, the names move together (with average correlations of c. 0.5), while across sleeves, they are less correlated (average long-short correlation of 0.24 and 0.11 in 2021-2026 and 2025-2026, respectively). Furthermore, the link between the long and the short baskets appears to be weakening in recent times, going from 0.43 in 2021-2026 to 0.23 in 2025-2026.

The correlations above indicate that the short sleeve hedges only the exposure the long and the short basket share, most likely general market exposure and exposure to tech, and less of it in 2025-2026. However, these correlations do not indicate that the short sleeve hedges the spread between hardware and software, which is the trade Contextual Alertness is trying to make.

## 1.4

> For each window in 1.1, report the standard error of the 4x book's annualized mean and of its Sharpe ratio, and the Sharpe ratio's 95% band. You are reviewing the book on June 30, 2026 with the statistics from 1.1 to 1.4 in hand. Which window would you base a risk assessment on, and what is a Sharpe ratio estimated on eighteen months of daily data evidence of? Answer in a short paragraph that uses the standard errors.
>
> ### Estimation Error
> Under independence, the standard error of an annualized mean estimated from $N$ years of daily data is $\sigma_{\text{ann}}/\sqrt{N}$. With $q = 252$ periods a year, the annualized Sharpe ratio has standard error $\sqrt{(1 + \text{SR}^2/2q)/N}$.

In [39]:
# 1.4
def estimation_error(r):
    N = len(r) / 252                                 
    ann_mean = r.mean() * 252
    ann_vol = r.std() * np.sqrt(252)
    sharpe = ann_mean / ann_vol

    se_mean = ann_vol / np.sqrt(N)
    se_sharpe = np.sqrt((1 + sharpe**2 / (2 * 252)) / N)

    return pd.Series({
        'Years (N)': N,
        'Ann. mean': ann_mean,
        'SE mean': se_mean,
        'Sharpe': sharpe,
        'SE Sharpe': se_sharpe,
        'Sharpe 95% low': sharpe - 1.96 * se_sharpe,
        'Sharpe 95% high': sharpe + 1.96 * se_sharpe,
    })

se_table = pd.DataFrame({'4x, 2021–2026': estimation_error(book_4x[FULL]),
                         '4x, 2025–2026': estimation_error(book_4x[RECENT])})
se_table.style.format('{:.2f}')

,"4x, 2021–2026","4x, 2025–2026"
Years (N),5.47,1.48
Ann. mean,1.25,4.25
SE mean,0.38,1.12
Sharpe,1.39,3.11
SE Sharpe,0.43,0.83
Sharpe 95% low,0.55,1.48
Sharpe 95% high,2.23,4.74


**Answer 1.4.**

We would base the risk assessment on the full window, as the standard errors for the mean and the Sharpe are around a third and half as large as the ones for the recent window (0.38 vs 1.12 for the mean and 0.43 vs 0.83 for the Sharpe, respectively). Furthermore, it includes 2021 to 2023, a regime in which the strategy lost money.

The Sharpe ratio estimated on the 18 months of data, 3.11, is evidence that the trade was working during that regime, but not necessarily an indication of the strategy's expected Sharpe. While it is high, its SE is also high (95% band of 1.48 to 4.74, vs 0.55 to 2.23 for the full window), and even higher if we consider the possibility that returns are not iid.

---
# 2. The Reset

## 2.1

> Start on 2025-01-02 with $100 million of NAV and positions set to the 4x target weights at the prior close. Hold cash as collateral without interest. Compare three rules:
>
> - Reset to target weights at every close.
> - Reset at every month-end close.
> - Hold the initial shares without rebalancing.
>
> Gross exposure is measured at each close for the following day. Report:
>
> - NAV and gross leverage through July 31, 2026, as plots.
> - For each rule: gross exposure set on June 22 and July 23, NAV relative to its initial value on June 22, the July 24 to 29 return, and whether and when NAV reaches zero.
> - For fixed shares initialized on 2021-01-04: maximum gross exposure and minimum NAV, with their dates.
> - For fixed shares initialized at the July 23, 2026 close to the 4x target on $100 million: NAV, long exposure, short exposure, and gross leverage at each of the four closes through July 29. Compare NAV with the daily-reset portfolio over the same four days.
>
> Could a fund have maintained each path? Use the gross exposures and NAV paths to explain what daily resetting assumes about the fund as a going concern, which convention a real fund lives under, and why the same positions lose different fractions of NAV over the reversal.

# 2.1

**Answer 2.1.**

---
# 3. One-Day VaR Through the Reversal

## 3.1

> Compute the EWMA volatility forecast of the 4x book from the first day your initialization allows to July 31, 2026, and, for every day in the design window, three one-day 5% VaR forecasts and the CVaR of each: historical simulation, Normal EWMA and filtered historical simulation.
>
> - Plot the EWMA forecast annualized; report the forecast made for July 24, 2026 and the fraction of forecasts for days through June 30, 2026 that were lower.
> - Report the three VaR and CVaR forecasts made at the July 23, 2026 close.
>
> ### VaR Conventions
>
> Historical simulation: minus the 5% quantile of the trailing 252 daily returns; the CVaR is minus the mean of the returns at or below that quantile. Normal EWMA: $\text{VaR}_t = -z_{0.05}\,\sigma_t$ and $\text{CVaR}_t = \sigma_t\,\phi(z_{0.05})/0.05$, with $\sigma_t$ the EWMA forecast. Filtered historical simulation: standardize each past return by the EWMA forecast that applied to it, $z_s = r_s/\sigma_s$; take the 5% quantile and the tail mean of the standardized returns over the trailing 252 days, the same window as the historical simulation; multiply both by the current forecast $\sigma_t$.

In [40]:
#3.1

**Answer 3.1.**

## 3.2

> Backtest the three methods over 2025-01-02 to 2026-07-23 and over 2021-01-04 to 2024-12-31. For each method and window report
>
> - the number of forecasts, the number of breaches and the hit ratio
> - the longest run of consecutive breaches
> - the mean realized loss on breach days beside the mean CVaR forecast on those days
>
> A method whose window has not yet filled starts later; report its count and say so. Which methods are calibrated in each window, and where the hit ratio is near 5%, does the breach-day loss match the CVaR forecast?

In [41]:
#3.2

**Answer 3.2.**

## 3.3

> For each of the four reversal days, July 24, 27, 28 and 29, report the realized loss of the 4x book as a multiple of the VaR each method had forecast for that day, and the four-day loss as a multiple of $\sqrt{4}$ times each method's July 23 forecast.
>
> Over 2025-01-02 to 2026-07-31, compare overlapping 4-day returns of the 4x book with $\sqrt{4}$ times the Normal EWMA one-day VaR in force at the start of each window. Report:
>
> - The hit ratio.
> - The largest ratio of realized loss to scaled VaR, with its date.
>
> For each method, count the reversal-day breaches. Does the four-day loss challenge $\sqrt{h}$ scaling? Say which of its assumptions (independent daily returns, constant volatility, a zero mean) could account for the gap between the scaled forecast and the realized loss, and what one four-day realization can and cannot establish about them. Is the largest one-day ratio on a reversal day?

In [42]:
#3.3

**Answer 3.3.**

## 3.4

> Using the July 23 EWMA volatility scaled by $\sqrt{4}$ and a zero mean, compute the probability under the normal of a four-day loss at least as large as the one realized. Compute it again by drawing four standardized returns at random from the 252-day FHS window, re-inflating them by the July 23 volatility, and compounding. Then recompute the FHS VaR and CVaR forecasts made at the July 23 close, and this probability, with a 504-day standardization window.
>
> - Report the probabilities, and the 504-day VaR and CVaR beside the 252-day ones.
> - Report the worst standardized return in each window, its date, and whether one draw of it reaches the realized loss.
>
> Say what that one observation does to the estimate, how far the conclusion depends on keeping it, and what each probability is a statement about: the model, the volatility level, or the book.

In [43]:
#3.4

**Answer 3.4.**

---
# 4. Where the Risk Sat

Section 4 uses daily excess returns and two estimation windows: 2021-01-04 to 2024-12-31 and 2025-01-02 to 2026-06-30. The 1x book keeps the betas readable; multiply by four for the 4x book.

## 4.1

> Regress each of the twelve names, the long sleeve (+50% of NAV at 1x), the short sleeve (-50%) and the 1x book on two factor sets: `SPY` alone and `SPY` with `SMH`.
>
> Report alpha, factor loadings, $R^2$ and residual volatility for the book and the two sleeves, annualized where appropriate. Use the individual-name regressions to verify the book's exposure vector $B\boldsymbol{w}$; a full table of individual regressions is not required.
>
> The book has zero net exposure. What is its market beta in each window, and which sleeve betas produce it? Did the short sleeve hedge the long sleeve's largest exposure, add to it, or leave it alone?
>
> ### Portfolio Exposures
>
> Regress each name's excess return on the factor excess returns, $\tilde r_{i,t} = \alpha_i + \boldsymbol{\beta}_i' \tilde{\boldsymbol{f}}_t + \epsilon_{i,t}$, and stack the loadings in $B$, one column per name. The book with weights $\boldsymbol{w}$ has factor exposure $B\boldsymbol{w}$ and residual return $\boldsymbol{w}'\boldsymbol{\epsilon}_t$.

In [44]:
#4.1

**Answer 4.1.**

## 4.2

> Estimate the 1x book's beta to `SPY` alone, and to `SPY` and `SMH` together, on a trailing one-year window at each quarter-end from September 2021 to June 2026. Tabulate or plot. The weights never changed. Did the exposures?

In [45]:
#4.2

**Answer 4.2.**

## 4.3

> For each factor set in 4.1 and each window, report the average pairwise correlation of the residuals among the long names, among the short names, and across the sleeves, next to the same averages for the raw returns. Then, for the 2025-01-02 to 2026-06-30 window, compute the 1x book's variance three ways: from the sample covariance of the twelve names; from the factor model with a diagonal residual covariance; from the factor model with the full residual covariance. Do this for `SPY` alone and for `SPY` plus `SMH`.
>
> - Convert each variance to a 4x one-day Normal 5% VaR.
>
> After the factors are removed, how many dimensions of risk are left in this book? Which term does the diagonal drop, and how large is it as a share of the book's variance?
>
> ### Variance Decomposition
>
> $\sigma^2_{\text{book}} = (B\boldsymbol{w})'\,\Sigma_f\,(B\boldsymbol{w}) + \boldsymbol{w}'\,\Sigma_\epsilon\,\boldsymbol{w}$. A factor risk model usually replaces $\Sigma_\epsilon$ with its diagonal; the sample covariance of the twelve names needs no such assumption.

In [46]:
#4.3

**Answer 4.3.**

## 4.4

> Attribute the July 24 to 29 loss of the 4x book to its factor exposure and its residual, as sums of daily contributions, using betas from each window and both factor sets from 4.1.
>
> - Report the systematic share under each of the four combinations.
>
> Was the reversal a factor event or a residual event, and does the answer depend on what you call a factor?

In [47]:
#4.4

**Answer 4.4.**

---
# 5. Horizon and Survival

## 5.1

> Compute daily log excess returns of the 1x and 4x books. For 2021-01-04 to 2024-12-31, 2025-01-02 to 2026-06-30 and 2021-01-04 to 2026-06-30, report
>
> - the annualized arithmetic mean, the annualized log mean, the annualized volatility, and the gap between the two means
> - for leverage 1x, 2x, 3x and 4x, the probability of underperforming cash over one month, one year and five years with the Barnstable formula, $\Pr\big[\texttt{r}_{t,t+h} < \texttt{r}^f_{t,t+h}\big] = \Phi\!\left(-\sqrt{h}\,\mu/\sigma\right)$, with $\mu$ and $\sigma$ the daily mean and standard deviation of the log excess return over cash
> - a plot of the probability against horizon for the four leverage levels on the 2021 to 2024 parameters
>
> What sign does the 4x log mean take in each window, and what produces the gap? At each leverage, does a longer horizon raise or lower the probability? State in one sentence how this relates to the argument Barnstable's trustees made about stocks.
>
> ### Log Returns
>
> Write $\texttt{r}_t = \ln(1+r_t)$ for the log return.

In [48]:
#5.1

**Answer 5.1.**

## 5.2

> Simulate the probability that the 4x book suffers a maximum drawdown of at least 50% within one year and within five years, using the 2021-01-04 to 2026-06-30 parameters: once from a normal with the book's daily log mean and standard deviation, once by resampling the book's own daily returns. State the number of paths and random seed.
>
> Compare the probabilities. Does the shape of the return distribution or the level of its volatility drive the answer? Explain how a drawdown from a running peak differs from a loss relative to initial wealth.
>
> For each simulated path, cumulate log returns from initial wealth of one, track the running maximum including that initial value, and record the largest drawdown.

In [49]:
#5.1

**Answer 5.2.**

## 5.3

> As of June 30, 2026, what gross exposure would you have recommended for this mandate, on what criterion, and estimated on which window? Give the number and the reasoning.

In [50]:
#5.3

**Answer 5.3.**

---
# 6. The Risk-Managed Mandate

The managed book keeps the twelve names, stays long the long sleeve and short the short sleeve, and never exceeds 4x gross. Every control uses only information available at the close before it acts. Every control is a rebalancing rule, and the constant-leverage comparators in 6.2 are daily-reset books at lower gross, so the comparison isolates timing from level.

## 6.1

> Before computing any Section 6 result, and before using any return after July 31, 2026, write a dated markdown cell that states
>
> - two numeric pass-fail criteria, one about risk and one about return or the cost of control
> - the control or controls you will test, chosen from a volatility target, a VaR or CVaR limit, a drawdown rule, a factor hedge, a sleeve or position cap, or a combination
> - every parameter
>
> One well-motivated control is sufficient. You may use the risk measure or hedge developed in Section 7 as part of it; those questions still require their own analysis. Keep the original declaration. Record any revision in a second dated cell.

In [51]:
#6.1

**Answer 6.1.**

## 6.2

> Implement the declared controls on the design window. Over the report window, 2025-01-02 to 2026-07-31, for the managed book, the unmanaged 4x book, the 1x book, and a constant-leverage comparator whose gross exposure equals the managed book's average gross over the report window, report
>
> - gross exposure through time and the dates on which the controls bound
> - cumulative return to July 23, the July 24 to 29 return, and cumulative return to July 31
> - annualized volatility, Sharpe ratio and maximum drawdown
> - average gross exposure, and the gross exposure set at the July 23 close (for July 24)
>
> Compare each control with constant exposure at the same average gross leverage to distinguish timing from position size. State what each control cost or earned before the reversal and during it, against the unmanaged book and against its comparator, and whether an investor reviewing the managed book in April 2026 would have kept it.

In [52]:
#6.2

**Answer 6.2.**

## 6.3

> Run the same controls, unchanged, from August 3, 2026 to the end of the file, with the same comparators.
>
> - Report the same statistics plus the worst day.
> - Apply the criteria declared in 6.1: pass or fail, by your own test.
>
> If the design window and the holdout disagree about a control, say which you believe and why.

In [53]:
#6.3

**Answer 6.3.**